In [1]:
!pip install motmetrics ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ultralytics] [ultralytics]


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from dataclasses import dataclass, field
from collections import defaultdict
from ultralytics import YOLO
from pathlib import Path
from matplotlib.patches import Rectangle
from scipy.optimize import linear_sum_assignment

import gc
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
def iou_matrix(a, b):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    b = np.asarray(b, dtype=float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=float)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    intersection = wh[..., 0] * wh[..., 1]
    area_a = np.clip(a[:, 2]-a[:, 0], 0, None) * np.clip(a[:, 3]-a[:, 1], 0, None)
    area_b = np.clip(b[:, 2]-b[:, 0], 0, None) * np.clip(b[:, 3]-b[:, 1], 0, None)
    union = area_a[:, None] + area_b[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)

def associate(track_boxes, det_boxes, min_iou=0.2):
    track_boxes = np.asarray(track_boxes, dtype=float).reshape(-1, 4)
    det_boxes = np.asarray(det_boxes, dtype=float).reshape(-1, 4)
    if len(track_boxes) == 0 or len(det_boxes) == 0:
        return [], list(range(len(track_boxes))), list(range(len(det_boxes)))
    ious = iou_matrix(track_boxes, det_boxes)
    rows, cols = linear_sum_assignment(1 - ious)
    matches = [(int(r), int(c)) for r, c in zip(rows, cols) if ious[r, c] >= min_iou]
    used_r = {r for r, _ in matches}; used_c = {c for _, c in matches}
    unmatched_tracks = [i for i in range(len(track_boxes)) if i not in used_r]
    unmatched_dets = [j for j in range(len(det_boxes)) if j not in used_c]
    return matches, unmatched_tracks, unmatched_dets

def run_tracker(tracker, detections):
    outputs = {}
    for frame in sorted(detections):
        outputs[frame] = tracker.update(detections[frame])
    return outputs

In [4]:
@dataclass
class Track:
    box: np.ndarray
    track_id: int
    velocity: np.ndarray = field(default_factory=lambda: np.zeros(2))
    age: int = 1
    hits: int = 1
    missed: int = 0
    last_center: np.ndarray | None = None

    def predict(self):
        shift = np.tile(self.velocity, 2)
        self.box = self.box + shift
        self.age += 1
        self.missed += 1
        return self.box.copy()

    def update(self, box):
        box = np.asarray(box, dtype=float)
        center = (box[:2] + box[2:]) / 2
        previous = (self.box[:2] + self.box[2:]) / 2
        measured_velocity = center - previous
        self.velocity = 0.65*self.velocity + 0.35*measured_velocity
        self.last_center = previous
        self.box = box.copy()
        self.hits += 1
        self.missed = 0


class OneStageTracker:
    def __init__(self, score_thresh=0.6, min_iou=0.15, max_age=2, min_hits=2):
        self.score_thresh = score_thresh
        self.min_iou = min_iou
        self.max_age = max_age
        self.min_hits = min_hits
        self.tracks = []
        self.next_id = 1

    def _new_track(self, box):
        self.tracks.append(Track(np.asarray(box, dtype=float).copy(), self.next_id))
        self.next_id += 1

    def visible(self):
        rows = [np.r_[tr.box, tr.track_id] for tr in self.tracks
                if tr.missed == 0 and tr.hits >= self.min_hits]
        return np.asarray(rows, dtype=float).reshape(-1, 5)


    def update(self, detections):
        detections = np.asarray(detections, dtype=float).reshape(-1, 5)
        predicted = np.asarray([tr.predict() for tr in self.tracks], dtype=float).reshape(-1, 4)
        selected = detections[detections[:, 4] >= self.score_thresh]
        matches, _, unmatched_dets = associate(predicted, selected[:, :4], self.min_iou)
        for track_idx, det_idx in matches:
            self.tracks[track_idx].update(selected[det_idx, :4])
        for det_idx in unmatched_dets:
            self._new_track(selected[det_idx, :4])
        self.tracks = [tr for tr in self.tracks if tr.missed <= self.max_age]
        return self.visible()

class ByteLiteTracker(OneStageTracker):
    def __init__(self, high_thresh=0.6, low_thresh=0.1, **kwargs):
        super().__init__(score_thresh=high_thresh, **kwargs)
        self.high_thresh = high_thresh
        self.low_thresh = low_thresh


    def update(self, detections):
        detections = np.asarray(detections, dtype=float).reshape(-1, 5)
        predicted = np.asarray([tr.predict() for tr in self.tracks], dtype=float).reshape(-1, 4)
        high = detections[detections[:, 4] >= self.high_thresh]
        low = detections[(detections[:, 4] >= self.low_thresh) &
                         (detections[:, 4] < self.high_thresh)]

        first, unmatched_tracks, unmatched_high = associate(predicted, high[:, :4], self.min_iou)
        for track_idx, det_idx in first:
            self.tracks[track_idx].update(high[det_idx, :4])

        remaining_boxes = predicted[unmatched_tracks] if unmatched_tracks else np.empty((0, 4))
        second_local, _, _ = associate(remaining_boxes, low[:, :4], self.min_iou)
        for local_track_idx, det_idx in second_local:
            track_idx = unmatched_tracks[local_track_idx]
            self.tracks[track_idx].update(low[det_idx, :4])

        for det_idx in unmatched_high:
            self._new_track(high[det_idx, :4])
        self.tracks = [tr for tr in self.tracks if tr.missed <= self.max_age]
        return self.visible()

In [5]:
def predict(model, paths, conf=0.05, imgsz=960):
    stream = model.predict(source=[str(path) for path in paths], stream=True,
                           conf=conf, imgsz=imgsz, classes=[0],
                           device=DEVICE, verbose=False, quantize="fp16")
    records = []
    for path, prediction in zip(paths, stream):
        frame = int(path.stem)
        if prediction.boxes is None or len(prediction.boxes) == 0:
            continue
        xyxy = prediction.boxes.xyxy.cpu().numpy()
        scores = prediction.boxes.conf.cpu().numpy()
        classes = prediction.boxes.cls.cpu().numpy().astype(int)
        for box, score, cls in zip(xyxy, scores, classes):
            records.append([frame, *box, float(score), int(cls)])
            
    table = pd.DataFrame(records, columns=["frame", "x1", "y1", "x2", "y2", "score", "class"])
    return table

def det_df_to_dict(det_df):
    """DataFrame -> {frame_id: ndarray (N,5) [x1,y1,x2,y2,score]}"""
    out = {}
    for fid, g in det_df.groupby("frame"):
        out[int(fid)] = g[["x1","y1","x2","y2","score"]].to_numpy(dtype=float)
    return out

def gt_to_dict(gt_df):
    """{frame_id: ndarray (N,5) [id, x1, y1, x2, y2]}"""
    out = {}
    for fid, g in gt_df.groupby("frame"):
        out[int(fid)] = g[["id", "x", "y", "x2", "y2"]].to_numpy(dtype=float)
    return out

def run_yolo_tracker(model, frame_paths, tracker="bytetrack.yaml", conf=0.05, imgsz=960):
    """Returns {frame_id: ndarray (N,5) [x1,y1,x2,y2,track_id]}"""
    model.predictor = None
    gc.collect()
    torch.cuda.empty_cache()

    stream = model.track(
        source=[str(p) for p in frame_paths], stream=True,
        tracker=tracker, conf=conf, imgsz=imgsz,
        classes=[0], device=DEVICE, verbose=False, persist=True, quantize="fp16"
    )
        
    outputs = {}
    try:
        for path, prediction in zip(frame_paths, stream):
            fid = int(path.stem)
            if prediction.boxes is None or prediction.boxes.id is None:
                outputs[fid] = np.empty((0, 5))
                continue
            
            xyxy = prediction.boxes.xyxy.cpu().numpy()
            ids = prediction.boxes.id.cpu().numpy().reshape(-1, 1)
            outputs[fid] = np.c_[xyxy, ids]
        return outputs
    finally:
        model.predictor = None
        gc.collect()
        torch.cuda.empty_cache()
    
def load_gt(annot_path):
    gt = pd.read_csv(annot_path, header=None,
                     names=["frame","id","x","y","w","h","conf","cls","vis","?"],
                     usecols=range(7))
    
    gt = gt[gt['conf'] >= 1]
    gt["x2"] = gt["x"] + gt["w"]
    gt["y2"] = gt["y"] + gt["h"]
    return gt


In [6]:
def hota_at_alpha(gt_dict, pred_dict, alpha):
    assoc_count = defaultdict(int)
    gt_lengths = defaultdict(int)
    pred_lengths = defaultdict(int)
    tpa = fpa = fna = 0

    for frame in sorted(set(gt_dict) | set(pred_dict)):
        gt_rows = gt_dict.get(frame,   np.empty((0, 5)))
        pred_rows = pred_dict.get(frame, np.empty((0, 5)))

        for row in gt_rows:   gt_lengths[int(row[0])] += 1
        for row in pred_rows: pred_lengths[int(row[4])] += 1

        gt_boxes = gt_rows[:, 1:5]  if len(gt_rows)   else np.empty((0, 4))
        pred_boxes = pred_rows[:, :4] if len(pred_rows)  else np.empty((0, 4))
        matches, unmatched_gt, unmatched_pred = associate(gt_boxes, pred_boxes, alpha)

        tpa += len(matches)
        fna += len(unmatched_gt)
        fpa += len(unmatched_pred)
        
        for gi, pi in matches:
            assoc_count[(int(gt_rows[gi, 0]), int(pred_rows[pi, 4]))] += 1

    det_a = tpa / max(tpa + fpa + fna, 1)
    ass_a = (
        sum(cnt / (gt_lengths[g] + pred_lengths[p] - cnt)
            for (g, p), cnt in assoc_count.items()) / tpa
        if tpa > 0 else 0.0
    )
    return det_a, ass_a


def compute_hota(gt_dict, pred_dict, alphas=np.arange(0.05, 0.96, 0.05)):
    deta_vals, assa_vals = zip(*[hota_at_alpha(gt_dict, pred_dict, a) for a in alphas])
    deta, assa = np.mean(deta_vals), np.mean(assa_vals)
    return {"HOTA": np.sqrt(deta * assa), "DetA": deta, "AssA": assa}


def evaluate_tracks(gt_dict, pred_dict, iou_thresh=0.5):
    import motmetrics as mm

    acc = mm.MOTAccumulator(auto_id=True)
    for frame in sorted(gt_dict):
        gt_rows = gt_dict.get(frame,   np.empty((0, 5)))
        pred_rows = pred_dict.get(frame, np.empty((0, 5)))

        gt_boxes = gt_rows[:, 1:5]  if len(gt_rows)   else np.empty((0, 4))
        pred_boxes = pred_rows[:, :4] if len(pred_rows)  else np.empty((0, 4))
        gt_ids = gt_rows[:, 0].astype(int).tolist()   if len(gt_rows)   else []
        pred_ids = pred_rows[:, 4].astype(int).tolist() if len(pred_rows) else []

        dist = 1.0 - iou_matrix(gt_boxes, pred_boxes) if (len(gt_boxes) and len(pred_boxes)) \
               else np.empty((len(gt_boxes), len(pred_boxes)))
        dist[dist > 1 - iou_thresh] = np.nan
        acc.update(gt_ids, pred_ids, dist)

    mh  = mm.metrics.create()
    mot = mh.compute(acc, metrics=["mota", "idf1", "num_switches", "precision", "recall"])

    return {
        "MOTA": float(mot["mota"].iloc[0]),
        "IDF1": float(mot["idf1"].iloc[0]),
        "IDSW": int(mot["num_switches"].iloc[0]),
        "precision": float(mot["precision"].iloc[0]),
        "recall": float(mot["recall"].iloc[0]),
        **compute_hota(gt_dict, pred_dict),
    }

In [8]:
DATA_DIR = '/kaggle/input/datasets/vadsorokin/drone-dataset'
# ANNOT_DIR = 'annotations'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


seq = "uav0000013_00000_v"
frame_paths = sorted(Path(DATA_DIR, seq, seq).glob('*.jpg'))

model = YOLO("yolo11n.pt")

det_df = predict(model, frame_paths, imgsz=960)
det_dict = det_df_to_dict(det_df)

# уравниваю наш трекер с тем, что из библиотеки
byte_lite_tracker = ByteLiteTracker(
    high_thresh=0.25,   # track_high_thresh
    low_thresh=0.1,     # track_low_thresh  
    min_iou=0.2,        # 1 - match_thresh (0.8)
    max_age=30,         # track_buffer
    min_hits=1,         # new_track_thresh 
)
byte_lite_out = run_tracker(byte_lite_tracker, det_dict)

yolo_tt_out = run_yolo_tracker(model, frame_paths, tracker="tracktrack.yaml", imgsz=960)
yolo_bt_out = run_yolo_tracker(model, frame_paths, tracker="bytetrack.yaml", imgsz=960)

gt_df = load_gt(str(Path(DATA_DIR, seq)) + '.txt')
gt = gt_to_dict(gt_df)

summary = pd.DataFrame({
    "ByteLite": evaluate_tracks(gt, byte_lite_out),
    "TrackTrack": evaluate_tracks(gt, yolo_tt_out),
    "ByteTrack": evaluate_tracks(gt, yolo_bt_out),
}).T.round(3)

display(summary)


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 331ms
Prepared 1 package in 76ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



,MOTA,IDF1,IDSW,precision,recall,HOTA,DetA,AssA
ByteLite,0.210,0.429,140.0,0.641,0.520,0.041,0.316,0.005
TrackTrack,0.093,0.176,0.0,0.931,0.100,0.019,0.078,0.005
ByteTrack,0.312,0.459,48.0,0.815,0.411,0.042,0.284,0.006


In [9]:
del det_df, det_dict, byte_lite_tracker, byte_lite_out, yolo_tt_out, yolo_bt_out
del model
gc.collect()
torch.cuda.empty_cache()


In [10]:
model = YOLO("yolo11n.pt")
det_df = predict(model, frame_paths, imgsz=[768, 1344])
det_dict = det_df_to_dict(det_df)

# уравниваю наш трекер с тем, что из библиотеки
byte_lite_tracker = byte_lite_tracker = ByteLiteTracker(
    high_thresh=0.25,   # track_high_thresh
    low_thresh=0.1,     # track_low_thresh  
    min_iou=0.2,        # 1 - match_thresh (0.8)
    max_age=30,         # track_buffer
    min_hits=1,         # new_track_thresh
)
byte_lite_out = run_tracker(byte_lite_tracker, det_dict)

yolo_tt_out = run_yolo_tracker(model, frame_paths, tracker="tracktrack.yaml", imgsz=[768, 1344])
yolo_bt_out = run_yolo_tracker(model, frame_paths, tracker="bytetrack.yaml", imgsz=[768, 1344])

summary = pd.DataFrame({
    "ByteLite": evaluate_tracks(gt, byte_lite_out),
    "TrackTrack": evaluate_tracks(gt, yolo_tt_out),
    "ByteTrack": evaluate_tracks(gt, yolo_bt_out),
}).T.round(3)

display(summary)

,MOTA,IDF1,IDSW,precision,recall,HOTA,DetA,AssA
ByteLite,-0.008,0.394,164.0,0.506,0.616,0.037,0.314,0.004
TrackTrack,0.235,0.385,10.0,0.910,0.262,0.037,0.199,0.007
ByteTrack,0.240,0.437,79.0,0.651,0.538,0.040,0.324,0.005
